# TP5 : diagnostiquer, régulariser, choisir l'optimiseur

Objectif : repartir du pipeline du TP4, provoquer un surapprentissage avec un CNN volontairement trop gros, le diagnostiquer dans TensorBoard, appliquer les remèdes un par un (dropout, batch norm, augmentation plus forte) en mesurant à chaque fois l'écart entre entraînement et validation, puis comparer les optimiseurs et un scheduler.

L'énoncé complet, avec les critères de réussite, est dans `enonce.md`.

*Complétez les cellules marquées `# TODO`, dans l'ordre.*

## Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

## Préparation (fournie)

**Données** : CIFAR-10 (Krizhevsky, Nair et Hinton, 2009), fourni par torchvision : des photos couleur de 32 x 32 pixels, en 10 classes (avion, automobile, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion). Le site du jeu ne donne pas de licence ; il demande de citer le rapport *Learning Multiple Layers of Features from Tiny Images* (Krizhevsky, 2009). Tous les TP utilisent les mêmes images : 5000 d'entraînement et 1000 de validation, tirées de l'entraînement de CIFAR-10, et 1000 de test.

Les images sont rangées comme dans un projet : un dossier `cifar10_fichiers/images/` avec une image PNG par fichier, et un fichier CSV par partie (`train.csv`, `val.csv`, `test.csv`) qui donne la classe de chaque image, sous forme de numéro.

In [ ]:
def preparer_archive_cifar10(racine="data"):
    """Place les fichiers de CIFAR-10 dans racine/cifar-10-batches-py sans passer par le serveur d'origine, lent.

    Dans l'ordre : fichiers déjà décompressés ; archive déposée dans la session (cifar-10-python.tar.gz ou
    cifar-10-batches-py.tar.gz, dans le dossier du notebook ou dans racine) ; archive d'origine copiée sur le
    Google Drive de la formation. Si rien n'aboutit, torchvision télécharge ensuite depuis le serveur d'origine ;
    dans tous les cas, il vérifie l'intégrité des fichiers.
    """
    import tarfile
    import urllib.request
    from pathlib import Path

    if Path(racine, "cifar-10-batches-py").exists():
        return
    noms = ("cifar-10-python.tar.gz", "cifar-10-batches-py.tar.gz")
    deposees = [Path(dossier, nom) for dossier in (".", racine) for nom in noms if Path(dossier, nom).exists()]
    if deposees:
        archive = deposees[0]
        print(f"Archive trouvée : {archive}")
    else:
        archive = Path(racine, "cifar-10-python.tar.gz")
        archive.parent.mkdir(parents=True, exist_ok=True)
        url = ("https://drive.usercontent.google.com/download?id=1kyq-1VhOPNr2Bo0w5DiLYuNghOdOWl0a"
               "&export=download&confirm=t")
        print("Téléchargement de CIFAR-10 depuis le Google Drive de la formation (170 Mo)...")
        try:
            urllib.request.urlretrieve(url, archive)
        except OSError as erreur:
            print(f"Google Drive indisponible ({erreur}) : téléchargement depuis le serveur d'origine.")
            return
    try:
        with tarfile.open(archive) as contenu:
            contenu.extractall(racine, filter="data")
    except tarfile.TarError:
        print(f"{archive} n'est pas une archive valide : téléchargement depuis le serveur d'origine.")


def charger_cifar10_fichiers(dossier="cifar10_fichiers", n_train=5000, n_val=1000, n_test=1000, seed=0):
    """Les images de charger_cifar10, rangées comme dans un projet : un dossier d'images et un CSV par partie.

    dossier/images/<partie>_<numero>.png, et dossier/<partie>.csv avec une ligne d'en-tête « fichier,classe »
    puis une ligne par image. La classe est un numéro : l'ordre alphabétique des noms français, le même
    qu'ImageFolder (automobile = 0, avion = 1, ..., oiseau = 9). Mêmes tirages que charger_cifar10. Le CSV
    d'une partie est écrit en dernier : une partie dont le CSV existe est complète et n'est pas réécrite. Les
    fichiers de CIFAR-10 sont d'abord récupérés par preparer_archive_cifar10, comme dans charger_cifar10.
    """
    from pathlib import Path

    import torch
    from torchvision.datasets import CIFAR10

    noms = ["avion", "automobile", "oiseau", "chat", "cerf", "chien", "grenouille", "cheval", "bateau", "camion"]
    numeros = {nom: numero for numero, nom in enumerate(sorted(noms))}
    generateur = torch.Generator().manual_seed(seed)
    ordre = torch.randperm(50000, generator=generateur).tolist()
    parties = {"train": (True, ordre[:n_train]),
               "val": (True, ordre[n_train:n_train + n_val]),
               "test": (False, torch.randperm(10000, generator=generateur)[:n_test].tolist())}
    preparer_archive_cifar10()
    Path(dossier, "images").mkdir(parents=True, exist_ok=True)
    for partie, (entrainement, indices) in parties.items():
        fichier_csv = Path(dossier, f"{partie}.csv")
        if fichier_csv.exists():
            continue
        source = CIFAR10("data", train=entrainement, download=True)
        lignes = ["fichier,classe"]
        for numero, indice in enumerate(indices):
            image, label = source[indice]
            nom_fichier = f"{partie}_{numero:05d}.png"
            image.save(Path(dossier, "images", nom_fichier))
            lignes.append(f"{nom_fichier},{numeros[noms[label]]}")
        fichier_csv.write_text("\n".join(lignes))

In [ ]:
charger_cifar10_fichiers()

## Étape 0 : le pipeline du TP4 (fourni)

Les classes et leurs effectifs, les transforms, puis le Dataset `ImagesCifar` écrit au TP4.

In [ ]:
from pathlib import Path

import torch

dossier = Path("cifar10_fichiers")
classes = ["automobile", "avion", "bateau", "camion", "cerf", "chat", "cheval", "chien", "grenouille", "oiseau"]
for partie in ("train", "val", "test"):
    lignes = (dossier / f"{partie}.csv").read_text().splitlines()[1:]
    effectifs = torch.bincount(torch.tensor([int(ligne.split(",")[1]) for ligne in lignes]), minlength=10)
    print(partie, len(lignes), effectifs.tolist())

In [ ]:
import torch
from torchvision.transforms import v2

conversion = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=[0.5] * 3, std=[0.5] * 3)]
eval_tf = v2.Compose(conversion)

In [ ]:
from PIL import Image
from torch.utils.data import Dataset


class ImagesCifar(Dataset):
    def __init__(self, fichier_csv, transform=None):
        self.dossier_images = Path(fichier_csv).parent / "images"
        self.transform = transform
        lignes = Path(fichier_csv).read_text().splitlines()[1:]
        self.exemples = [ligne.split(",") for ligne in lignes]

    def __len__(self):
        return len(self.exemples)

    def __getitem__(self, i):
        fichier, classe = self.exemples[i]
        image = Image.open(self.dossier_images / fichier)
        if self.transform is not None:
            image = self.transform(image)
        return image, int(classe)

Pour rendre le surapprentissage visible, on n'entraîne que sur 400 images tirées au hasard, une quarantaine par classe : c'est la situation d'un projet où les données étiquetées sont rares. La validation garde toutes ses images.

In [ ]:
from torch.utils.data import DataLoader, Subset

val_loader = DataLoader(ImagesCifar("cifar10_fichiers/val.csv", transform=eval_tf), batch_size=256)
torch.manual_seed(0)
indices_400 = torch.randperm(5000)[:400].tolist()


def loader_entrainement(transform):
    torch.manual_seed(0)
    return DataLoader(Subset(ImagesCifar("cifar10_fichiers/train.csv", transform=transform), indices_400),
                      batch_size=32, shuffle=True)

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    for x, y in loader:
        loss = loss_fn(model(x.to(device)), y.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

`fit` entraîne un modèle et enregistre ses courbes dans TensorBoard, sous `runs/tp5/<nom>`. Sans optimiseur donné, elle prend Adam (lr=1e-3) ; un scheduler, s'il est donné, avance d'un pas à chaque époque.

In [ ]:
from torch.utils.tensorboard import SummaryWriter


def fit(nom, model, train_loader, optimizer=None, scheduler=None, n_epochs=20):
    loss_fn = nn.CrossEntropyLoss()
    if optimizer is None:
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    writers = {"trn": SummaryWriter(f"runs/tp5/{nom}-trn"), "val": SummaryWriter(f"runs/tp5/{nom}-val")}
    history = {"loss_trn": [], "loss_val": [], "acc_trn": [], "acc_val": []}
    for epoch in range(n_epochs):
        train_one_epoch(model, train_loader, loss_fn, optimizer)
        if scheduler is not None:
            scheduler.step()
        for jeu, loader in (("trn", train_loader), ("val", val_loader)):
            loss, acc = evaluate(model, loader, loss_fn)
            history[f"loss_{jeu}"].append(loss)
            history[f"acc_{jeu}"].append(acc)
            writers[jeu].add_scalar("loss", loss, epoch)
    for writer in writers.values():
        writer.close()
    ecart = history["acc_trn"][-1] - history["acc_val"][-1]
    print(f"{nom} : acc trn {history['acc_trn'][-1]:.3f}  acc val {history['acc_val'][-1]:.3f}  écart {ecart:.3f}  "
          f"loss val min {min(history['loss_val']):.3f}, finale {history['loss_val'][-1]:.3f}")
    return history

## Étape 1 : provoquer le surapprentissage

La classe fournie construit un CNN volontairement trop gros pour 400 images, plus gros que celui du TP4 : quatre blocs de 32, 64, 128 puis 256 filtres, soit 2 x 2 avant le classifieur, et une couche cachée de 128 neurones. Ses options `batchnorm` et `dropout` serviront à l'étape 3 : lisez-la. `nouveau_cnn` crée le modèle avec une graine fixe, sur le device.

In [ ]:
class GrosCNN(nn.Module):
    def __init__(self, batchnorm=False, dropout=0.0):
        super().__init__()
        couches = []
        for entree, sortie in [(3, 32), (32, 64), (64, 128), (128, 256)]:
            couches.append(nn.Conv2d(entree, sortie, kernel_size=3, padding=1))
            if batchnorm:
                couches.append(nn.BatchNorm2d(sortie))
            couches += [nn.ReLU(), nn.MaxPool2d(2)]
            if dropout > 0:
                couches.append(nn.Dropout2d(p=dropout))
        self.features = nn.Sequential(*couches)
        tete = [nn.Flatten(), nn.Linear(256 * 2 * 2, 128), nn.ReLU()]
        if dropout > 0:
            tete.append(nn.Dropout(p=0.5))
        tete.append(nn.Linear(128, 10))
        self.classifieur = nn.Sequential(*tete)

    def forward(self, x):
        return self.classifieur(self.features(x))


def nouveau_cnn(batchnorm=False, dropout=0.0):
    torch.manual_seed(0)
    return GrosCNN(batchnorm, dropout).to(device)

In [ ]:
assert nouveau_cnn()(torch.randn(2, 3, 32, 32).to(device)).shape == (2, 10)

Entraînez-le 20 époques, sans option et sans augmentation : une ligne, avec `fit`.

In [ ]:
# TODO : fit("sur-parametre", nouveau_cnn(), loader_entrainement(eval_tf)), rangé dans histories
histories = ...

## Étape 2 : diagnostiquer

Ouvrez TensorBoard, puis tracez les courbes (cellules fournies). À quelle époque la loss de validation est-elle minimale ? Que fait-elle ensuite ?

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/tp5

In [ ]:
import matplotlib.pyplot as plt


def tracer(histories):
    for nom, h in histories.items():
        ligne, = plt.plot(h["loss_trn"], label=f"{nom} (trn)")
        plt.plot(h["loss_val"], "--", color=ligne.get_color(), label=f"{nom} (val)")
    plt.xlabel("époque")
    plt.ylabel("loss")
    plt.legend(fontsize=8)
    plt.show()

In [ ]:
tracer(histories)

*Votre diagnostic :*

## Étape 3 : régulariser, un remède à la fois

**3a.** Dropout (`dropout=0.25`).

In [ ]:
# TODO : nouveau_cnn(dropout=0.25), sans augmentation, sous le nom "dropout"

**3b.** Dropout et batch norm.

In [ ]:
# TODO : nouveau_cnn(batchnorm=True, dropout=0.25), sans augmentation, sous le nom "dropout-bn"

**3c.** Dropout, batch norm et augmentation plus forte : rotation de 10 degrés, recadrage aléatoire (`RandomResizedCrop`, entre 80 et 100 % de l'image), miroir horizontal. Avant de lancer : ces images transformées ressemblent-elles encore à de vraies photos de chaque classe ?

In [ ]:
train_tf_fort = v2.Compose([
    v2.RandomRotation(degrees=10),
    # TODO : v2.RandomResizedCrop(32, scale=(0.8, 1.0), antialias=True)
    # TODO : v2.RandomHorizontalFlip(p=0.5)
] + conversion)

In [ ]:
# TODO : nouveau_cnn(batchnorm=True, dropout=0.25) avec train_tf_fort, sous le nom "dropout-bn-aug"

In [ ]:
tracer(histories)

## Étape 4 : choisir l'optimiseur

On garde le modèle régularisé (batch norm, dropout, augmentation), entraîné jusqu'ici avec Adam. Comparez trois autres réglages, en créant à chaque fois l'optimiseur sur les paramètres du nouveau modèle :

- SGD avec momentum : `torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)` ;
- AdamW : `torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)` ;
- AdamW avec un scheduler `CosineAnnealingLR(optimizer, T_max=20)`, qui fait décroître le learning rate jusqu'à la vingtième époque.

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

histories_optim = {"Adam": histories["+ augmentation"]}

model = nouveau_cnn(batchnorm=True, dropout=0.25)
# TODO : SGD avec momentum, lr=0.01, momentum=0.9
optimizer = ...
histories_optim["SGD momentum"] = fit("sgd", model, loader_entrainement(train_tf_fort), optimizer)

model = nouveau_cnn(batchnorm=True, dropout=0.25)
# TODO : AdamW, lr=1e-3, weight_decay=0.01
optimizer = ...
histories_optim["AdamW"] = fit("adamw", model, loader_entrainement(train_tf_fort), optimizer)

model = nouveau_cnn(batchnorm=True, dropout=0.25)
# TODO : AdamW comme ci-dessus, puis CosineAnnealingLR(optimizer, T_max=20)
optimizer = ...
scheduler = ...
histories_optim["AdamW + cosine"] = fit("adamw-cosine", model, loader_entrainement(train_tf_fort), optimizer,
                                        scheduler)

In [ ]:
tracer(histories_optim)

## Étape 5 : bilan (fourni)

Quel remède a le plus réduit l'écart ? Quel réglage d'optimiseur donne la meilleure validation ?

In [ ]:
for nom, h in {**histories, **histories_optim}.items():
    print(f"{nom:16s} écart final {h['acc_trn'][-1] - h['acc_val'][-1]:.3f}  acc val {h['acc_val'][-1]:.3f}  "
          f"loss val finale {h['loss_val'][-1]:.3f}")

*Votre bilan :*

## Bonus

**B1. Garder le meilleur modèle.** Gardez une copie des poids à l'époque où la loss de validation est minimale, et rechargez-la à la fin.

In [ ]:
import copy

model = nouveau_cnn()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_loader = loader_entrainement(eval_tf)
meilleure_loss, meilleurs_poids = float("inf"), None
for epoch in range(20):
    train_one_epoch(model, train_loader, nn.CrossEntropyLoss(), optimizer)
    loss_val, _ = evaluate(model, val_loader, nn.CrossEntropyLoss())
    # TODO : si loss_val < meilleure_loss, mémoriser loss_val et copy.deepcopy(model.state_dict())
    ...
model.load_state_dict(meilleurs_poids)
print(f"loss val du modèle rechargé : {evaluate(model, val_loader, nn.CrossEntropyLoss())[0]:.3f}")